<a href="https://colab.research.google.com/github/dan-zeman/belo-horizonte/blob/main/Udapi1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Universal Dependencies with `udapi-python`

This notebook introduces students of computational linguistics to the `udapi-python` library, a powerful tool for working with Universal Dependencies treebanks. We will cover installation, downloading a treebank, and performing a basic search for linguistic features.

## 1. Install `udapi`

First, we need to install the `udapi` Python library. This can be done using `pip`.

In [1]:
%%bash
# (Preceding a single line with ! makes that line interpreted by shell instead of python.
# Inserting %%bash at the beginning of the cell makes the whole cell interpreted by bash.)
pip install --upgrade udapi
udapy -h

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 418.7/418.7 kB 9.9 MB/s eta 0:00:00
usage: udapy [optional_arguments] scenario

udapy - Python interface to Udapi - API for Universal Dependencies

Examples of usage:
  udapy -s read.Sentences udpipe.En < in.txt > out.conllu
  udapy -T < sample.conllu | less -R
  udapy -HAM ud.MarkBugs < sample.conllu > bugs.html

positional arguments:
  scenario              A sequence of blocks and their parameters.

options:
  -h, --help            show this help message and exit
  -q, --quiet           Warning, info and debug messages are suppressed. Only fatal errors are reported.
  -v, --verbose         Warning, info and debug messages are printed to the STDERR.
  -s, --save            Add write.Conllu to the end of the scenario
  -T, --save_text_mode_trees
                        Add write.TextModeTrees color=1 to the end of the scenario
  -H, --save_html       Add write.TextModeTreesHtml color=1 to the end of the scenario
  -A, --save_all_attributes
 

## 2. Download and Load a UD Treebank

Udapi allows easy access to Universal Dependencies treebanks. As an example, we will download several treebanks from the parallel UD collection. Feel free to add downloads of other treebanks you are interested in. See https://universaldependencies.org/ for the list of available treebanks. Besides the UD homepage, you can also try https://universaldependencies.org/languages.html, where you can filter languages by family and genus.

In [2]:
%%bash
rm -rf UD_*
#git clone https://github.com/UniversalDependencies/UD_Czech-PUD.git
#git clone https://github.com/UniversalDependencies/UD_English-PUD.git
#git clone https://github.com/UniversalDependencies/UD_Spanish-PUD.git
#git clone https://github.com/UniversalDependencies/UD_Portuguese-PUD.git
git clone https://github.com/UniversalDependencies/UD_Czech-FicTree.git
git clone https://github.com/UniversalDependencies/UD_Portuguese-Porttinari.git
git clone https://github.com/UniversalDependencies/UD_Hungarian-Szeged.git

Cloning into 'UD_Czech-FicTree'...
Cloning into 'UD_Portuguese-Porttinari'...
Cloning into 'UD_Hungarian-Szeged'...


Now we will access the treebank data from Python. The `udapi.core.document.Document` class is used to load the treebank. You can edit the code cell below and specify the treebank you want to work with (it must be one of the treebanks you downloaded above).

In [3]:
import glob
from udapi.core.document import Document

# Read the CoNLL-U file.
###############################################################################
# TODO: Change this variable to explore different datasets.
treebank = 'UD_Hungarian-Szeged' # Student can modify this line.

# List all .conllu files in the specified treebank folder.
conllu_files = glob.glob(f"{treebank}/*.conllu")
print(f"Found {len(conllu_files)} CoNLL-U files in {treebank}:")
for file in sorted(conllu_files):
    print(file)
print(f"Loading {treebank}...")
# Each CoNLL-U file will be stored as one Document object.
# Note that some treebanks use the '# newdoc' comment to mark document boundaries within each file.
# That is a different notion of document!
filedocs = []
nsent = 0
for file in sorted(conllu_files):
    doc = Document(filename=file)
    filedocs.append(doc)
    nsent += len(doc.bundles)
print(f"Loaded {nsent} sentences from {treebank}.")


Found 3 CoNLL-U files in UD_Hungarian-Szeged:
UD_Hungarian-Szeged/hu_szeged-ud-dev.conllu
UD_Hungarian-Szeged/hu_szeged-ud-test.conllu
UD_Hungarian-Szeged/hu_szeged-ud-train.conllu
Loading UD_Hungarian-Szeged...
Loaded 1800 sentences from UD_Hungarian-Szeged.


## 3. Find and Print Lemmas of Personal Pronouns

Now, let's explore the loaded treebank. We'll iterate through each word (node) in every sentence (bundle) in every CoNLL-U file of the treebank and identify personal pronouns (UPOS tag `PRON` and PronType feature `Prs`). Then, we'll print their lemmas.

In [4]:
personal_pronoun_lemmas = set()

for doc in filedocs:
    # Bundle is a unit that corresponds to one sentence in Udapi.
    # In theory it may have multiple trees for that sentence; but normally there is just one.
    for bundle in doc.bundles:
        root = bundle.get_tree()
        nodes = root.descendants
        for node in nodes:
            # Check if the word is a pronoun (UPOS tag 'PRON')
            # and if its PronType feature is 'Prs' (Personal Pronoun)
            if node.upos == 'PRON' and node.feats['PronType'] == 'Prs':
                personal_pronoun_lemmas.add(node.lemma)

print(f"\nLemmas of personal pronouns found in {treebank}:")
for lemma in sorted(list(personal_pronoun_lemmas)):
    print(lemma)


Lemmas of personal pronouns found in UD_Hungarian-Szeged:
alatt
bele
belőle
ellen
ellene
elébe
előtte
fölött
helyett
hozzá
iránt
körülötte
közt
közte
közötte
közül
közüle
maga
mellett
mi
neki
néki
nélkül
rajta
rá
róla
saját
szerint
szerinte
számára
te
tőle
vele
által
általa
én
ön
önmaga
ő
ők


### Exercise

The following code cell is similar to the previous one but it is more general, intended as a template for simple searches. Look for the TODO comments! They mark places where you should complete the code. (You cannot just run the cell as it is now. It will not work.)

In [ ]:
# We will collect the results of our query in the list called "results".
results = []

for doc in filedocs:
    # Bundle is a unit that corresponds to one sentence in Udapi.
    # In theory it may have multiple trees for that sentence; but normally there is just one.
    for bundle in doc.bundles:
        root = bundle.get_tree()
        nodes = root.descendants
        for node in nodes:
            ###############################################################################
            # TODO: Now it is time to set the requirements for the nodes we are looking for.
            # Modify the condition on the next line to reflect your search criteria.
            # Note that "equals to" is expressed by "==" (two equal symbols, not just one),
            # "is not equal to" is expressed by "!=". You can combine multiple conditions
            # using the logical operators "and", "or", "not" (and round brackets if needed).
            # You can query various attributes of the current node:
            # node.ord ... the integer identifying position of the word in the sentence (column ID in CoNLL-U)
            # node.form ... the word form (column FORM in CoNLL-U)
            # node.lemma ... the lemma of the word (column LEMMA in CoNLL-U)
            # node.upos ... the universal POS tag (column UPOS in CoNLL-U)
            # node.feats['FFFF'] ... FFFF is a name of a feature, for example node.feats['Gender']
            # node.deprel ... the label of the incoming relation from the parent to this node (column DEPREL in CoNLL-U)
            # node.misc['MMMM'] ... MMMM is a name of an attribute in the MISC column.
            #     The inventories of attributes are specific to each treebank. An example of
            #     an attribute that is available in multiple treebanks and can be useful is
            #     'Translit' (transcription/transliteration from a foreign script) and 'Gloss'
            #     (translation of the current word to English or another major language).
            if node.XXXX == 'YYYY':
                ###############################################################################
                # TODO: What do we want to remember from the node as a result? Perhaps the lemma?
                # Or upos? Or a feature? Any node attribute that could be used in the requirements
                # (see above) could also be used as the result. Replace the XXXX below.
                results.append(node.XXXX)

###############################################################################
# TODO: Remember, Grew Match shows 1000 results at most. Here we can set our
# limit:
max_results_to_show = 10
n = len(results)
print(f"Treebank {treebank}\nFound {n} results:")
# Discard the results that are over the limit.
if n > max_results_to_show:
    results = results[:max_results_to_show]
for result in results:
    print(result)


## 4. Some Technical Code to Enable Clustering Tables

You don't need to understand or modify the following cell but you must run it so that we can display nice clustering tables later.

In [26]:
from collections import defaultdict
import pandas as pd

def generate_two_dimensional_table(clusters, display_mode='#'):
    """
    Generates, sorts, and displays a two-dimensional table from a clusters dictionary.

    Args:
        clusters (dict): A dictionary where keys are (parent_tag, child_tag) tuples
                         and values are counts.
        display_mode (str): The mode for displaying cell values.
                            Options: '#': absolute counts,
                                     'H%': horizontal percentage (row-wise),
                                     'V%': vertical percentage (column-wise),
                                     '%': percentage of grand total.
    Returns:
        pandas.DataFrame: The DataFrame used for display.
    """
    import pandas as pd

    # Get all unique parent and child tags
    parent_tags = sorted(list(set([pair[0] for pair in clusters.keys()])))
    child_tags = sorted(list(set([pair[1] for pair in clusters.keys()])))

    # Create a DataFrame to store the counts (initial data only)
    df_raw_counts = pd.DataFrame(0, index=parent_tags, columns=child_tags)

    # Populate the DataFrame with counts from the clusters dictionary
    for (parent, child), count in clusters.items():
        df_raw_counts.loc[parent, child] = count

    # Calculate row sums for sorting purposes
    row_sums_for_sorting = df_raw_counts.sum(axis=1)

    # Sort rows based on these sums in descending order
    sorted_row_indices = row_sums_for_sorting.sort_values(ascending=False).index
    df_raw_counts = df_raw_counts.reindex(sorted_row_indices)

    # Calculate column sums for sorting purposes (after row sorting)
    column_sums_for_sorting = df_raw_counts.sum(axis=0)

    # Sort columns based on these sums in descending order
    sorted_column_names = column_sums_for_sorting.sort_values(ascending=False).index
    df_raw_counts = df_raw_counts.reindex(columns=sorted_column_names)

    # --- Prepare DataFrame for display based on display_mode ---
    df_display = df_raw_counts.copy() # Start with raw counts

    # Calculate absolute totals for later use (these will always be absolute counts)
    absolute_row_totals = df_raw_counts.sum(axis=1)
    absolute_column_totals = df_raw_counts.sum(axis=0)
    absolute_grand_total = df_raw_counts.sum().sum()

    if display_mode == 'H%':
        # Horizontal percentage: percent of the total count of the row
        df_display = df_raw_counts.div(absolute_row_totals.replace(0, 1), axis=0) * 100
        df_display = df_display.round(2).fillna(0)
        df_display = df_display.map(lambda x: f'{x:.2f}%')
    elif display_mode == 'V%':
        # Vertical percentage: percent of the total count of the column
        df_display = df_raw_counts.div(absolute_column_totals.replace(0, 1), axis=1) * 100
        df_display = df_display.round(2).fillna(0)
        df_display = df_display.map(lambda x: f'{x:.2f}%')
    elif display_mode == '%':
        # Percent of the total count in the table
        df_display = (df_raw_counts / absolute_grand_total) * 100
        df_display = df_display.round(2).fillna(0)
        df_display = df_display.map(lambda x: f'{x:.2f}%')
    elif display_mode == '#':
        # Absolute counts (already in df_raw_counts, convert to string for display consistency)
        df_display = df_raw_counts.astype(str)
    else:
        raise ValueError("Invalid display_mode. Choose from '#', 'H%', 'V%', '%'.")

    # Add the absolute Row_Total column
    df_display['Total'] = absolute_row_totals.astype(str)

    # Add the absolute Col_Total row
    col_total_row_for_display = absolute_column_totals.astype(str)
    col_total_row_for_display['Total'] = str(absolute_grand_total) # Grand total
    df_display.loc['Total'] = col_total_row_for_display

    return df_display

## 5. Query Properties of the Node and Its Parent

So far we were only looking at the properties of one node (although one of the properties was `node.deprel`, the label of the incoming dependency relation from its parent). We can query all these properties also for the parent of the current node. For example, `node.parent.upos` denotes the UPOS category of the parent. Let's examine the UPOS tags of the parent and the child of a particular relation type.

In [27]:
# clusters will store: {(parent UPOS, child UPOS): count}
clusters = defaultdict(int)

for doc in filedocs:
    # Bundle is a unit that corresponds to one sentence in Udapi.
    # In theory it may have multiple trees for that sentence; but normally there is just one.
    for bundle in doc.bundles:
        root = bundle.get_tree()
        nodes = root.descendants
        for node in nodes:
            ###################################################################
            # TODO: You can modify your search criteria here.
            if node.deprel == 'nsubj':
                ###############################################################
                # TODO: You can modify your clustering keys here. For example,
                # instead of upos, you could collect the lemma of the node and
                # its parent.
                # Get the pair of UPOS tags (first parent, then child).
                upospair = (node.parent.upos, node.upos)
                clusters[upospair] += 1

# Now display the clusters as a table.

###############################################################################
# TODO: Set your preferred display mode here.
# Options: '#':  absolute counts
#          'H%': horizontal percentage (percent of the total count of the row)
#          'V%': vertical percentage (percent of the total count of the column)
#          '%':  percent of the total count in the table
display_mode = '#' # Change this variable to select the desired display mode

# Generate the table of clusters.
df_display = generate_two_dimensional_table(clusters, display_mode=display_mode)

# Display the table
print(f"Parent UPOS (rows) vs. Child UPOS (columns) for 'nsubj' dependency (Mode: {display_mode}):")
display(df_display)


Parent UPOS (rows) vs. Child UPOS (columns) for 'nsubj' dependency (Mode: #):


,NOUN,PROPN,PRON,ADJ,NUM,ADV,DET,Total
VERB,1525,345,266,17,9,5,1,2168
ADJ,161,19,35,2,1,0,0,218
NOUN,101,14,46,4,2,0,0,167
PRON,20,1,3,2,0,0,0,26
PROPN,16,2,0,0,0,0,0,18
ADV,6,4,3,0,0,0,0,13
AUX,1,2,1,0,0,0,0,4
DET,1,0,2,0,0,0,0,3
NUM,0,0,1,0,0,0,0,1
Total,1831,387,357,25,12,5,1,2618


In [28]:
# @title Download the table for further analysis
from google.colab import files

# Define the filename for the CSV file
output_csv_filename = 'clusters.tsv'

# Save the DataFrame to a tab-separated CSV file
df_display.to_csv(output_csv_filename, sep='\t', encoding='utf-8')

print(f"File '{output_csv_filename}' is ready for download.")
files.download(output_csv_filename)

File 'clusters.tsv' is ready for download.


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## 6. Collect All Forms of All Lemmas

This will help us to use various criteria to select and display paradigms of individual lexemes.

In [29]:
# dictionary will store: {UPOS: {lemma: {form: {featstring: count}}}}
dictionary = defaultdict(lambda: defaultdict(lambda: defaultdict(lambda: defaultdict(int))))
all_lemmas = {}
all_forms = {}

for doc in filedocs: # Iterate through each Document in the list of loaded CoNLL-U files
    for bundle in doc.bundles: # Iterate through bundles within each Document
        root = bundle.get_tree()
        nodes = root.descendants
        for node in nodes:
            # Exclude nodes marked as typos
            if not (node.feats and node.feats.get('Typo') == 'Yes'):
                upos = node.upos
                lemma = node.lemma.lower()
                form = node.form.lower()
                # The feature ExtPos, if present, is not interesting.
                # It is not about morphology but about syntax of the sentence.
                # Note: This will erase the feature from the loaded document, so we will not be able to work with it. It will however not destroy it in the file on the disk.
                node.feats['ExtPos'] = ''
                featstring = str(node.feats) if node.feats else 'NO_FEATS'
                dictionary[upos][lemma][form][featstring] += 1
                all_lemmas[lemma] = True
                all_forms[form] = True

print(f"Collected {len(all_forms)} distinct forms and {len(all_lemmas)} distinct lemmas in total.")
print(f"This makes the average morphological richness {len(all_forms)/len(all_lemmas)} forms per lemma.")

Collected 13468 distinct forms and 8782 distinct lemmas in total.
This makes the average morphological richness 1.5335914370302892 forms per lemma.


In [30]:
# @title Select the paradigm with most forms and print it.
import pandas as pd

maxforms = 0
winners = []
for upos in dictionary:
    uposdict = dictionary[upos]
    for lemma in uposdict:
        lemmadict = uposdict[lemma]
        nforms = len(lemmadict)
        # We can choose to consider only one UPOS category.
        #if upos != 'NOUN':
        #    continue
        if nforms > maxforms:
            maxforms = nforms
            winners = [(upos, lemma)]
        elif nforms == maxforms:
            winners.append((upos, lemma))
print(f"Maximum number of forms found in one lexeme is {maxforms}.")
print(f"It was observed with the following UPOS + LEMMA combinations:")
for upos, lemma in winners:
    print(f"{upos} {lemma}")

# Change this to == 1 if you only want to see unique winners.
if len(winners) >= 1:
    print()
    selected_upos = winners[0][0]
    selected_lemma = winners[0][1]
    lemmadict = dictionary[selected_upos][selected_lemma]

    # Prepare data for a DataFrame
    table_data = []
    for form in sorted(lemmadict):
        for featstring in sorted(lemmadict[form]):
            table_data.append({
                'Form': form,
                'Features': featstring,
                'Count': lemmadict[form][featstring]
            })

    df_paradigm = pd.DataFrame(table_data)
    print(f"Paradigm for {selected_upos} '{selected_lemma}':")

    # Manually format for left-aligned first two columns and right-aligned last column
    if not df_paradigm.empty:
        # Calculate max widths for 'Form' and 'Features' columns
        max_form_width = max(len(str(x)) for x in df_paradigm['Form'].tolist() + ['Form'])
        max_features_width = max(len(str(x)) for x in df_paradigm['Features'].tolist() + ['Features'])
        max_count_width = max(len(str(x)) for x in df_paradigm['Count'].tolist() + ['Count'])

        # Print header
        print(f"{('Form').ljust(max_form_width)} {('Features').ljust(max_features_width)} {('Count').rjust(max_count_width)}")
        # Print separator
        print(f"{'-'*max_form_width} {'-'*max_features_width} {'-'*max_count_width}")

        # Print data rows
        for index, row in df_paradigm.iterrows():
            lemmamark = ' <=== LEMMA' if row['Form'] == selected_lemma else ''
            print(f"{str(row['Form']).ljust(max_form_width)} {str(row['Features']).ljust(max_features_width)} {str(row['Count']).rjust(max_count_width)}{lemmamark}")


Maximum number of forms found in one lexeme is 20.
It was observed with the following UPOS + LEMMA combinations:
PRON az

Paradigm for PRON 'az':
Form    Features                                   Count
------- ------------------------------------------ -----
abba    Case=Ill|Number=Sing|Person=3|PronType=Dem     1
abban   Case=Ine|Number=Sing|Person=3|PronType=Dem    11
abból   Case=Ela|Number=Sing|Person=3|PronType=Dem     1
addig   Case=Ter|Number=Sing|Person=3|PronType=Dem     1
ahhoz   Case=All|Number=Sing|Person=3|PronType=Dem     9
annak   Case=Dat|Number=Sing|Person=3|PronType=Dem     7
annak   Case=Gen|Number=Sing|Person=3|PronType=Dem    29
arra    Case=Sbl|Number=Sing|Person=3|PronType=Dem    36
arról   Case=Del|Number=Sing|Person=3|PronType=Dem    20
attól   Case=Abl|Number=Sing|Person=3|PronType=Dem    10
avval   Case=Ins|Number=Sing|Person=3|PronType=Dem     1
az      Case=Nom|Number=Sing|Person=3|PronType=Dem    75 <=== LEMMA
azok    Case=Nom|Number=Plur|Person=3|PronTyp

## Tips for other things to show

Run the commandline interface to Udapi, capture its HTML output, then either show it in a notebook cell or let the user download it.

In [31]:
from IPython.display import HTML
from google.colab import files

# Capture HTML output from a shell command in a variable.
###!!! Saving this as udapi_output.html will not work because it mixes STDOUT with STDERR, and the latter is not even in HTML.
#html_output = !cat UD_Czech-PUD/cs_pud-ud-test.conllu | udapy -HAM util.Mark node='node.upos == "PRON"'
!cat UD_Czech-PUD/cs_pud-ud-test.conllu | udapy -HAM util.Mark node='node.upos == "PRON"' > udapi_output.html

# html_output is a list of strings, we have to join them into one string.
#html_content = "\n".join(html_output)

# Show the HTML content in a notebook cell.
#HTML(html_content)

# Save the HTML content in a file in the virtual machine.
output_filename = 'udapi_output.html'
#with open(output_filename, 'w') as f:
#    f.write(html_content)

# Offer the file for download.
print(f"File '{output_filename}' is ready for download.")
files.download(output_filename)


cat: UD_Czech-PUD/cs_pud-ud-test.conllu: No such file or directory
2026-07-22 02:32:01,959 [   INFO] run_blocks - No reader specified, using read.Conllu
2026-07-22 02:32:01,959 [   INFO] run_blocks -  ---- ROUND ----
2026-07-22 02:32:01,960 [   INFO] run_blocks - Executing block read.Conllu {}
2026-07-22 02:32:01,960 [   INFO] try_fast_load - Reading -
2026-07-22 02:32:01,960 [   INFO] run_blocks - Executing block util.Mark node=node.upos == "PRON"
2026-07-22 02:32:01,960 [   INFO] run_blocks - Executing block write.TextModeTreesHtml color=1 attributes=form,lemma,upos,xpos,feats,deprel,misc marked_only=1
File 'udapi_output.html' is ready for download.


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>